In [2]:
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions


In [3]:

df = pd.read_csv("train.csv")

# convertir cada fila en una frase descriptiva
def fila_a_texto(row):
    return (
        f"Orden {row['Order ID']} del {row['Order Date']}: "
        f"Cliente {row['Customer Name']} ({row['Segment']}) "
        f"compró {row['Product Name']} de la categoría {row['Category']} / {row['Sub-Category']} "
        f"por ${row['Sales']:.2f} en {row['City']}, {row['State']} ({row['Region']})."
    )

documentos = df.apply(fila_a_texto, axis=1).tolist()
ids = [f"row_{i}" for i in range(len(documentos))]

# indexar en ChromaDB
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
cliente_db = chromadb.PersistentClient(path="./mi_base_vectorial")

# borrar colección anterior y crear nueva con los datos reales
try:
    cliente_db.delete_collection("mis_documentos")
except:
    pass

coleccion = cliente_db.get_or_create_collection(
    name="mis_documentos",
    embedding_function=embedding_fn
)

# indexar en lotes de 500 (ChromaDB tiene límite por lote)
batch_size = 500
for i in range(0, len(documentos), batch_size):
    coleccion.add(
        documents=documentos[i:i+batch_size],
        ids=ids[i:i+batch_size]
    )
    print(f"✅ Indexados {min(i+batch_size, len(documentos))}/{len(documentos)}")

print(f"\n🎉 Listo — {len(documentos)} filas indexadas")

c:\Users\JHONATAN\anaconda3\envs\data_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4118.81it/s]


✅ Indexados 500/9800
✅ Indexados 1000/9800
✅ Indexados 1500/9800
✅ Indexados 2000/9800
✅ Indexados 2500/9800
✅ Indexados 3000/9800
✅ Indexados 3500/9800
✅ Indexados 4000/9800
✅ Indexados 4500/9800
✅ Indexados 5000/9800
✅ Indexados 5500/9800
✅ Indexados 6000/9800
✅ Indexados 6500/9800
✅ Indexados 7000/9800
✅ Indexados 7500/9800
✅ Indexados 8000/9800
✅ Indexados 8500/9800
✅ Indexados 9000/9800
✅ Indexados 9500/9800
✅ Indexados 9800/9800

🎉 Listo — 9800 filas indexadas
